# Merge and finalize cell labels
- Finalize label columns
- Merge the labels from integration of the Xenium CD4 cells with the CITE-seq CD4 cells with the larger labeled adata object
- Save cell type palettes

**Pinned Environment:** [`conda_envs/space2_20250604.yml`](../conda_envs/space2_20250604.yml)  

In [ ]:
from pathlib import Path
import sys
import os
import warnings

import numpy as np
import pandas as pd
import seaborn as sns
import anndata
import scanpy as sc
import squidpy as sq

from scipy.sparse import csr_matrix

import pickle
import random
import colorcet

In [ ]:
warnings.simplefilter(action='ignore', category=Warning)

## Local file info


In [ ]:
sys.path.append(str(Path.cwd().resolve().parents[0]))

from config.paths import BASE_OUTDIR, INPUTS_DIR, FUNCTIONS_DIR

inputs_dir = INPUTS_DIR
out_dir = os.path.join(BASE_OUTDIR, "merge_labels")
labels_dir = os.path.join(BASE_OUTDIR, "cell_labeling/ouputs_mouselung_hurskainen_ref_consolidate_labels_xenseg_HDM")
seurat_outs_dir = os.path.join(labels_dir, "cite_integration/seurat_obj_outs/cite_xen_integrated")

print(inputs_dir)
print(out_dir)

if not os.path.exists(out_dir):
    os.makedirs(out_dir)

plot_out_dir = os.path.join(out_dir, 'plots')
if not os.path.exists(plot_out_dir):
    os.makedirs(plot_out_dir)

## Load adata

In [ ]:
# labeled adata object
adata = sc.read_h5ad(os.path.join(labels_dir, "adata_labeled.h5ad"))

## Load integrated CD4s with CITE, from seurat object

In [ ]:
# Load data
counts = pd.read_csv(os.path.join(seurat_outs_dir, "counts.csv"), index_col=0)
meta = pd.read_csv(os.path.join(seurat_outs_dir, "cell_metadata.csv"), index_col=0)

# Transpose counts so rows = cells, columns = genes
counts_t = counts.T

# Build AnnData
adata_int = sc.AnnData(X=csr_matrix(counts_t.values))
adata_int.obs = meta
adata_int.var_names = counts_t.columns
adata_int.obs_names = counts_t.index

# add integrated umap dims
umap = pd.read_csv(os.path.join(seurat_outs_dir, "umap.csv"), index_col=0)
adata_int.obsm["X_umap_int"] = umap.loc[adata_int.obs_names].values

adata_int_xen = adata_int[adata_int.obs['modality']=='xen', :]

## Main adata object label updates
- Set label_scavni_refine to label_fine and add coarser label categories
- Update label_fine with CD4 subset labels from CITE integration

In [ ]:
# set label fine 
adata.obs["label_fine"] = adata.obs["label_scanvi_refine"].astype(str)  

# Add integrated labels
cd4_cells = adata.obs_names.intersection(adata_int_xen.obs_names)
adata.obs.loc[cd4_cells, "label_fine"] = adata_int_xen.obs.loc[cd4_cells, "integrated_cluster"]
adata.obs["label_fine"] = adata.obs["label_fine"].astype('category')  

# Load the CSV mapping file
mapping_df = pd.read_csv(os.path.join(inputs_dir, '20250519_mouse_lung_label_mapping_ordered.csv'))

# Convert mapping_df into a dictionary for fast lookup
mapping_dict = mapping_df.set_index("label_fine")[["label_medium", "label_coarse", "label_coarser"]].to_dict(orient="index")

# Apply mapping to create new columns in adata.obs
adata.obs["label_medium"] = adata.obs["label_fine"].map(lambda x: mapping_dict.get(x, {}).get("label_medium"))
adata.obs["label_coarse"] = adata.obs["label_fine"].map(lambda x: mapping_dict.get(x, {}).get("label_coarse"))
adata.obs["label_coarser"] = adata.obs["label_fine"].map(lambda x: mapping_dict.get(x, {}).get("label_coarser"))

adata.obs.drop(columns=["label_scanvi_refine"], inplace=True)

# add timepoint column for easier plotting
adata.obs.loc[adata.obs['sample_label']=='HDM_day3', "timepoint"] = 'day3'
adata.obs.loc[adata.obs['sample_label']=='HDM_day30', "timepoint"] = 'day30'

In [ ]:
# set order of labels so that major cell types are together

# label_fine order
fine_order = mapping_df["label_fine"].dropna().unique().tolist()
adata.obs["label_fine"] = pd.Categorical(adata.obs["label_fine"], categories=fine_order, ordered=True)

# label_medium order
medium_order = mapping_df["label_medium"].dropna().unique().tolist()
adata.obs["label_medium"] = pd.Categorical(adata.obs["label_medium"], categories=medium_order, ordered=True)

# label_coarse order
coarse_order = mapping_df["label_coarse"].dropna().unique().tolist()
adata.obs["label_coarse"] = pd.Categorical(adata.obs["label_coarse"], categories=coarse_order, ordered=True)

# label_coarser order
coarser_order = mapping_df["label_coarser"].dropna().unique().tolist()
adata.obs["label_coarser"] = pd.Categorical(adata.obs["label_coarser"], categories=coarser_order, ordered=True)


## Cell type label palettes

In [ ]:
sc.pl.embedding(adata, basis = 'X_umap', color='label_fine')
sc.pl.embedding(adata, basis = 'X_umap', color='label_medium')

celltype_palette = adata.uns['label_fine_colors']
celltype_palette_mapped_original = {val: color for val, color in zip(adata.obs['label_fine'].cat.categories, celltype_palette)}
celltype_palette_mapped = celltype_palette_mapped_original.copy()

In [ ]:
# palette updates

# T cell subsets
celltype_palette_mapped['CD4 trans'] = '#ddefff' 
celltype_palette_mapped['Th0'] = 'darkred' 
celltype_palette_mapped['Th17'] = 'goldenrod' 
celltype_palette_mapped['CD4 trans'] = 'cornflowerblue' 
celltype_palette_mapped['Treg'] = 'darkolivegreen' 

In [ ]:
sc.pl.embedding(adata, basis = 'X_umap', color='label_fine', palette = celltype_palette_mapped)

In [ ]:
# label coarse palette
palette_coarser = sns.color_palette(colorcet.glasbey, 
                                n_colors=20)
palette_coarser = random.sample(palette_coarser, k=len(palette_coarser))
celltype_palette_coarser_mapped = {val: color for val, color in zip(adata.obs['label_coarser'].cat.categories, palette_coarser)}

sc.pl.embedding(adata, basis = 'X_umap_scanvi_refalign', color='label_coarser', palette = celltype_palette_coarser_mapped)

celltype_palette_coarse_mapped = {val: color for val, color in zip(adata.obs['label_coarse'].cat.categories, palette_coarser)}
sc.pl.embedding(adata, basis = 'X_umap_scanvi_refalign', color='label_coarse', palette = celltype_palette_coarse_mapped)

### Save palettes
Save palettes to repo input dir so they accessible in repo.

In [ ]:
with open(os.path.join(inputs_dir, 'palettes/20250520_celltype_palette_coarser_mapped.pkl'), 'wb') as f:
    pickle.dump(celltype_palette_coarser_mapped, f)

with open(os.path.join(inputs_dir, 'palettes/20250520_celltype_palette_coarse_mapped.pkl'), 'wb') as f:
    pickle.dump(celltype_palette_coarse_mapped, f)

with open(os.path.join(inputs_dir, 'palettes/20250519_celltype_palette_mapped.pkl'), 'wb') as f:
    pickle.dump(celltype_palette_mapped, f)

## Make separate adata_cd4 with integrated umap dimensions

In [ ]:
adata_cd4 = adata[adata.obs['label_medium']=='CD4 act', :]

In [ ]:
# Find shared cells
cd4_cells = adata_cd4.obs_names.intersection(adata_int_xen.obs_names)

# Subset UMAP coordinates by matching cell indices
adata_cd4.obsm['X_umap_int'] = adata_int_xen.obsm['X_umap_int'][adata_int_xen.obs_names.get_indexer(cd4_cells)]

## Save adata and adata_cd4

In [ ]:
adata.write_h5ad(os.path.join(out_dir,'adata_labeled.h5ad'), compression='gzip')
adata_cd4.write_h5ad(os.path.join(out_dir,'adata_labeled_cd4.h5ad'), compression='gzip')
adata_int.write_h5ad(os.path.join(out_dir,'adata_cd4_int.h5ad'), compression='gzip')

In [ ]:
import session_info
print('active conda environment: ', os.path.basename(sys.prefix))
session_info.show(excludes=['google3'])